In [1]:
# ============================================================
# OLIST E-COMMERCE DATASET - COMPLETE EDA
# EDA ONLY
# NO ENCODING
# NO SCALING
# NO TRAIN/TEST SPLIT
# NO MACHINE LEARNING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from scipy.stats import (
    chi2_contingency,
    levene,
    normaltest
)

from sklearn.preprocessing import PowerTransformer

warnings.filterwarnings("ignore")




In [2]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("olist_merged_dataset.csv")

print("\n" + "=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Shape:", df.shape)





DATASET LOADED
Shape: (113425, 18)


In [3]:
# ============================================================
# 2. CREATE OUTPUT FOLDER
# ============================================================

os.makedirs("EDA_Plots", exist_ok=True)




In [4]:
# ============================================================
# 3. DATASET OVERVIEW
# ============================================================

print("\n========== HEAD ==========")
print(df.head())

print("\n========== TAIL ==========")
print(df.tail())

print("\n========== SAMPLE ==========")
print(
    df.sample(
        min(5, len(df)),
        random_state=42
    )
)

print("\n========== SHAPE ==========")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())





========== HEAD ==========
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-2

In [5]:
# ============================================================
# 4. DATA INFORMATION
# ============================================================

print("\n========== DATA INFORMATION ==========")

df.info()

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("\nNumerical Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)





========== DATA INFORMATION ==========
<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  str    
 1   customer_id                    113425 non-null  str    
 2   order_status                   113425 non-null  str    
 3   order_purchase_timestamp       113425 non-null  str    
 4   order_approved_at              113264 non-null  str    
 5   order_delivered_carrier_date   111457 non-null  str    
 6   order_delivered_customer_date  110196 non-null  str    
 7   order_estimated_delivery_date  113425 non-null  str    
 8   customer_unique_id             113425 non-null  str    
 9   customer_zip_code_prefix       113425 non-null  int64  
 10  customer_city                  113425 non-null  str    
 11  customer_state                 113425 non-null  str    
 12  o

In [6]:
# ============================================================
# 5. MISSING VALUES
# ============================================================

print("\n========== MISSING VALUES ==========")

missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": df.isnull().mean() * 100
})

missing = missing[
    missing["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

print(missing)

if missing.empty:
    print("No missing values found.")





========== MISSING VALUES ==========
                               Missing Values  Percentage
order_delivered_customer_date            3229    2.846815
order_delivered_carrier_date             1968    1.735067
seller_id                                 775    0.683271
order_item_id                             775    0.683271
product_id                                775    0.683271
price                                     775    0.683271
shipping_limit_date                       775    0.683271
freight_value                             775    0.683271
order_approved_at                         161    0.141944


In [7]:
# ============================================================
# 6. MISSING VALUE TREATMENT
# ============================================================

print("\n========== MISSING VALUE TREATMENT ==========")

# Numerical columns -> Median
for col in numeric_cols:

    if df[col].isnull().sum() > 0:

        median_value = df[col].median()

        df[col] = df[col].fillna(
            median_value
        )

        print(
            col,
            "-> filled with median:",
            median_value
        )


# Categorical columns -> Mode
for col in categorical_cols:

    if df[col].isnull().sum() > 0:

        mode_value = df[col].mode()

        if len(mode_value) > 0:

            df[col] = df[col].fillna(
                mode_value.iloc[0]
            )

            print(
                col,
                "-> filled with mode:",
                mode_value.iloc[0]
            )


print(
    "Remaining missing values:",
    df.isnull().sum().sum()
)





========== MISSING VALUE TREATMENT ==========
order_item_id -> filled with median: 1.0
price -> filled with median: 74.99
freight_value -> filled with median: 16.26
order_approved_at -> filled with mode: 2018-02-24 03:20:27
order_delivered_carrier_date -> filled with mode: 2018-05-09 15:48:00
order_delivered_customer_date -> filled with mode: 2017-07-31 18:03:02
product_id -> filled with mode: aca2eb7d00ea1a7b8ebd4e68314663af
seller_id -> filled with mode: 6560211a19b47992c3666cc44a7e94c0
shipping_limit_date -> filled with mode: 2017-07-21 18:25:23
Remaining missing values: 0


In [8]:
# ============================================================
# 7. BLANK VALUES
# ============================================================

print("\n========== BLANK VALUES ==========")

blank_found = False

for col in categorical_cols:

    blank = (
        df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    if blank > 0:

        blank_found = True

        print(
            col,
            ":",
            blank
        )

if not blank_found:

    print("No blank values found.")





========== BLANK VALUES ==========
No blank values found.


In [9]:
# ============================================================
# 8. DUPLICATES
# ============================================================

print("\n========== DUPLICATES ==========")

duplicate_rows = df.duplicated().sum()

print(
    "Duplicate rows:",
    duplicate_rows
)

if duplicate_rows > 0:

    df = df.drop_duplicates()

    print(
        "Duplicate rows removed."
    )

else:

    print(
        "No complete duplicate rows found."
    )





========== DUPLICATES ==========
Duplicate rows: 0
No complete duplicate rows found.


In [10]:
# ============================================================
# 9. DUPLICATE ID CHECK
# ============================================================

print("\n========== DUPLICATE ID CHECK ==========")

for col in [
    "order_id",
    "customer_id",
    "product_id",
    "seller_id"
]:

    if col in df.columns:

        print(
            f"Duplicate {col}:",
            df[col].duplicated().sum()
        )





========== DUPLICATE ID CHECK ==========
Duplicate order_id: 13984
Duplicate customer_id: 13984
Duplicate product_id: 80474
Duplicate seller_id: 110330


In [11]:
# ============================================================
# 10. UNIQUE VALUES
# ============================================================

print("\n========== UNIQUE VALUES ==========")

for col in df.columns:

    print(
        col,
        "->",
        df[col].nunique()
    )





========== UNIQUE VALUES ==========
order_id -> 99441
customer_id -> 99441
order_status -> 8
order_purchase_timestamp -> 98875
order_approved_at -> 90733
order_delivered_carrier_date -> 81018
order_delivered_customer_date -> 95664
order_estimated_delivery_date -> 459
customer_unique_id -> 96096
customer_zip_code_prefix -> 14994
customer_city -> 4119
customer_state -> 27
order_item_id -> 21
product_id -> 32951
seller_id -> 3095
shipping_limit_date -> 93318
price -> 5968
freight_value -> 6999


In [12]:
# ============================================================
# 11. VALUE COUNTS
# ============================================================

print("\n========== VALUE COUNTS ==========")

for col in [
    "order_status",
    "customer_state",
    "customer_city"
]:

    if col in df.columns:

        print(
            f"\n--- {col} ---"
        )

        print(
            df[col]
            .value_counts()
            .head(15)
        )





========== VALUE COUNTS ==========

--- order_status ---
order_status
delivered      110197
shipped          1186
canceled          706
unavailable       610
invoiced          361
processing        357
created             5
approved            3
Name: count, dtype: int64

--- customer_state ---
customer_state
SP    47820
RJ    14669
MG    13220
RS     6269
PR     5787
SC     4201
BA     3821
DF     2421
GO     2346
ES     2264
PE     1810
CE     1487
PA     1085
MT     1059
MA      831
Name: count, dtype: int64

--- customer_city ---
customer_city
sao paulo                17946
rio de janeiro            7885
belo horizonte            3167
brasilia                  2407
curitiba                  1762
campinas                  1669
porto alegre              1619
salvador                  1419
guarulhos                 1340
sao bernardo do campo     1070
niteroi                    984
santo andre                895
osasco                     854
santos                     827
goiania    

In [13]:
# ============================================================
# 12. DESCRIPTIVE STATISTICS
# ============================================================

print("\n========== DESCRIPTIVE STATISTICS ==========")

print(
    df[numeric_cols].describe()
)





========== DESCRIPTIVE STATISTICS ==========
       customer_zip_code_prefix  order_item_id          price  freight_value
count             113425.000000  113425.000000  113425.000000  113425.000000
mean               35102.472965       1.196482     120.341732      19.964832
std                29864.919733       0.702900     183.044145      15.755309
min                 1003.000000       1.000000       0.850000       0.000000
25%                11250.000000       1.000000      39.900000      13.080000
50%                24320.000000       1.000000      74.990000      16.260000
75%                59020.000000       1.000000     134.000000      21.150000
max                99990.000000      21.000000    6735.000000     409.680000


In [14]:
# ============================================================
# 13. MEDIAN
# ============================================================

print("\n========== MEDIAN ==========")

print(
    df[numeric_cols].median()
)





========== MEDIAN ==========
customer_zip_code_prefix    24320.00
order_item_id                   1.00
price                          74.99
freight_value                  16.26
dtype: float64


In [15]:
# ============================================================
# 14. MODE
# ============================================================

print("\n========== MODE ==========")

print(
    df[numeric_cols]
    .mode()
    .iloc[0]
)





========== MODE ==========
customer_zip_code_prefix    22790.0
order_item_id                   1.0
price                          59.9
freight_value                  15.1
Name: 0, dtype: float64


In [16]:
# ============================================================
# 15. GROUPING ANALYSIS
# ============================================================

print("\n========== GROUPING ANALYSIS ==========")

group_status = df.groupby(
    "order_status"
).agg(

    Orders=(
        "order_id",
        "nunique"
    ),

    Average_Price=(
        "price",
        "mean"
    ),

    Average_Freight=(
        "freight_value",
        "mean"
    ),

    Total_Sales=(
        "price",
        "sum"
    )
)

print(group_status)





========== GROUPING ANALYSIS ==========
              Orders  Average_Price  Average_Freight  Total_Sales
order_status                                                     
approved           2      69.866667        10.493333       209.60
canceled         625     152.313924        18.862734    107533.63
created            5      74.990000        16.260000       374.95
delivered      96478     119.980563        19.948598  13221498.11
invoiced         314     170.848615        20.761496     61676.35
processing       301     169.297535        25.083725     60439.22
shipped         1107     127.152133        22.275008    150802.43
unavailable      609      77.420754        16.291115     47226.66


In [17]:
# ============================================================
# 16. STATE ANALYSIS
# ============================================================

print("\n========== STATE ANALYSIS ==========")

state_analysis = df.groupby(
    "customer_state"
).agg(

    Orders=(
        "order_id",
        "nunique"
    ),

    Customers=(
        "customer_unique_id",
        "nunique"
    ),

    Average_Price=(
        "price",
        "mean"
    ),

    Total_Sales=(
        "price",
        "sum"
    )
)

print(
    state_analysis
    .sort_values(
        "Total_Sales",
        ascending=False
    )
    .head(15)
)





========== STATE ANALYSIS ==========
                Orders  Customers  Average_Price  Total_Sales
customer_state                                               
SP               41746      40302     109.384700   5230776.34
RJ               12852      12384     124.810265   1830841.77
MG               11635      11259     120.433595   1592132.12
RS                5466       5277     120.091511    752853.68
PR                5045       4882     118.646672    686608.29
SC                3637       3534     124.358031    522428.09
BA                3380       3277     134.257987    512999.77
DF                2140       2075     125.455923    303728.79
GO                2020       1952     125.987562    295566.82
ES                2033       1964     121.747893    275637.23
PE                1652       1609     145.352481    263087.99
CE                1336       1313     153.281520    227929.62
PA                 975        949     165.274433    179322.76
MT                 907        87

In [18]:
# ============================================================
# 17. COMPARISON OF MEANS
# ============================================================

print("\n========== COMPARISON OF MEANS ==========")

mean_comparison = df.groupby(
    "order_status"
)[
    [
        "price",
        "freight_value"
    ]
].mean()

print(mean_comparison)





========== COMPARISON OF MEANS ==========
                   price  freight_value
order_status                           
approved       69.866667      10.493333
canceled      152.313924      18.862734
created        74.990000      16.260000
delivered     119.980563      19.948598
invoiced      170.848615      20.761496
processing    169.297535      25.083725
shipped       127.152133      22.275008
unavailable    77.420754      16.291115


In [19]:
# ============================================================
# 18. VARIANCE
# ============================================================

print("\n========== VARIANCE CHECK ==========")

variance = df[numeric_cols].var()

print(variance)





========== VARIANCE CHECK ==========
customer_zip_code_prefix    8.919134e+08
order_item_id               4.940682e-01
price                       3.350516e+04
freight_value               2.482298e+02
dtype: float64


In [20]:
# ============================================================
# 19. STANDARD DEVIATION
# ============================================================

print("\n========== STANDARD DEVIATION ==========")

std = df[numeric_cols].std()

print(std)





========== STANDARD DEVIATION ==========
customer_zip_code_prefix    29864.919733
order_item_id                   0.702900
price                         183.044145
freight_value                  15.755309
dtype: float64


In [21]:
# ============================================================
# 20. LEVENE VARIANCE TEST
# ============================================================

print("\n========== LEVENE VARIANCE TEST ==========")

price_groups = []

for status, group in df.groupby(
    "order_status"
):

    values = (
        group["price"]
        .dropna()
    )

    if len(values) > 1:

        price_groups.append(
            values
        )


if len(price_groups) >= 2:

    levene_stat, levene_p = levene(
        *price_groups,
        center="median"
    )

    print(
        "Levene Statistic:",
        levene_stat
    )

    print(
        "P-value:",
        levene_p
    )

    if levene_p < 0.05:

        print(
            "Result: Variances are significantly different."
        )

    else:

        print(
            "Result: No significant difference in variances."
        )





========== LEVENE VARIANCE TEST ==========
Levene Statistic: 26.366104053210723
P-value: 2.272788409311778e-36
Result: Variances are significantly different.


In [22]:
# ============================================================
# 21. CHI-SQUARE INDEPENDENCE TEST
# ============================================================

print("\n========== CHI-SQUARE INDEPENDENCE TEST ==========")

table = pd.crosstab(
    df["order_status"],
    df["customer_state"]
)

chi2, p_value, dof, expected = (
    chi2_contingency(table)
)

print(
    "Chi-Square:",
    chi2
)

print(
    "P-value:",
    p_value
)

print(
    "Degrees of Freedom:",
    dof
)

if p_value < 0.05:

    print(
        "Result: Variables are statistically associated."
    )

else:

    print(
        "Result: No significant association detected."
    )





========== CHI-SQUARE INDEPENDENCE TEST ==========
Chi-Square: 637.2063354835925
P-value: 7.877783955135938e-52
Degrees of Freedom: 182
Result: Variables are statistically associated.


In [23]:
# ============================================================
# 22. CRAMER'S V
# ============================================================

print("\n========== CRAMER'S V ==========")

n = table.to_numpy().sum()

phi2 = chi2 / n

rows, columns = table.shape

cramers_v = np.sqrt(
    phi2 /
    min(
        columns - 1,
        rows - 1
    )
)

print(
    "Cramer's V:",
    cramers_v
)





========== CRAMER'S V ==========
Cramer's V: 0.028329351213186495


In [24]:
# ============================================================
# 23. NORMALITY TEST
# ============================================================

print("\n========== NORMALITY TEST ==========")

print(
    "D'Agostino-Pearson Normality Test"
)

normality_results = []

normal_sample = df.sample(
    min(5000, len(df)),
    random_state=42
)

for col in numeric_cols:

    values = (
        normal_sample[col]
        .dropna()
    )

    if len(values) >= 20:

        statistic, p = normaltest(
            values
        )

        normality_results.append([
            col,
            statistic,
            p
        ])

        if p < 0.05:

            result = (
                "Not normally distributed"
            )

        else:

            result = (
                "Approximately normal"
            )

        print(
            f"{col}: p={p:.6f} -> {result}"
        )


normality_df = pd.DataFrame(
    normality_results,
    columns=[
        "Column",
        "Statistic",
        "P-value"
    ]
)

print(
    "\nNormality Test Summary:"
)

print(normality_df)





========== NORMALITY TEST ==========
D'Agostino-Pearson Normality Test
customer_zip_code_prefix: p=0.000000 -> Not normally distributed
order_item_id: p=0.000000 -> Not normally distributed
price: p=0.000000 -> Not normally distributed
freight_value: p=0.000000 -> Not normally distributed

Normality Test Summary:
                     Column    Statistic        P-value
0  customer_zip_code_prefix   837.029824  1.742943e-182
1             order_item_id  7369.892476   0.000000e+00
2                     price  6129.556860   0.000000e+00
3             freight_value  5714.141572   0.000000e+00


In [25]:
# ============================================================
# 24. SKEWNESS BEFORE TREATMENT
# ============================================================

print("\n========== SKEWNESS BEFORE TREATMENT ==========")

skew_before = df[
    numeric_cols
].skew()

print(skew_before)

print("""
Interpretation:
Around 0 = approximately symmetric
Positive = right skewed
Negative = left skewed
""")

print(
    "\nHighly skewed variables:"
)

for col, value in skew_before.items():

    if abs(value) > 1:

        print(
            col,
            "-> Highly skewed:",
            round(value, 4)
        )

    elif abs(value) > 0.5:

        print(
            col,
            "-> Moderately skewed:",
            round(value, 4)
        )

    else:

        print(
            col,
            "-> Approximately symmetric:",
            round(value, 4)
        )





========== SKEWNESS BEFORE TREATMENT ==========
customer_zip_code_prefix    0.781055
order_item_id               7.605870
price                       7.950387
freight_value               5.660770
dtype: float64

Interpretation:
Around 0 = approximately symmetric
Positive = right skewed
Negative = left skewed


Highly skewed variables:
customer_zip_code_prefix -> Moderately skewed: 0.7811
order_item_id -> Highly skewed: 7.6059
price -> Highly skewed: 7.9504
freight_value -> Highly skewed: 5.6608


In [26]:
# ============================================================
# 25. KURTOSIS BEFORE TREATMENT
# ============================================================

print("\n========== KURTOSIS BEFORE TREATMENT ==========")

kurt_before = df[
    numeric_cols
].kurt()

print(kurt_before)

print("""
Approximately 0 = normal-like tails
Positive = heavier tails
Negative = lighter tails
""")





========== KURTOSIS BEFORE TREATMENT ==========
customer_zip_code_prefix     -0.791527
order_item_id               104.535362
price                       121.629093
freight_value                60.208747
dtype: float64

Approximately 0 = normal-like tails
Positive = heavier tails
Negative = lighter tails



In [27]:
# ============================================================
# 26. OUTLIER DETECTION - IQR
# ============================================================

print("\n========== OUTLIER DETECTION - IQR ==========")

continuous_cols = [
    "price",
    "freight_value"
]

outlier_table = []

for col in continuous_cols:

    Q1 = df[col].quantile(
        0.25
    )

    Q3 = df[col].quantile(
        0.75
    )

    IQR = Q3 - Q1

    lower = (
        Q1 - 1.5 * IQR
    )

    upper = (
        Q3 + 1.5 * IQR
    )

    count = (
        (df[col] < lower) |
        (df[col] > upper)
    ).sum()

    percentage = (
        count /
        df[col].notna().sum()
    ) * 100

    outlier_table.append([
        col,
        Q1,
        Q3,
        IQR,
        lower,
        upper,
        count,
        percentage
    ])

outlier_df = pd.DataFrame(
    outlier_table,
    columns=[
        "Column",
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Outlier Count",
        "Outlier %"
    ]
)

print(outlier_df)





========== OUTLIER DETECTION - IQR ==========
          Column     Q1      Q3    IQR  Lower Bound  Upper Bound  \
0          price  39.90  134.00  94.10     -101.250      275.150   
1  freight_value  13.08   21.15   8.07        0.975       33.255   

   Outlier Count  Outlier %  
0           8465   7.463081  
1          12134  10.697818  


In [28]:
# ============================================================
# 27. OUTLIER TREATMENT
# ============================================================

print("\n========== OUTLIER TREATMENT ==========")

for col in continuous_cols:

    Q1 = df[col].quantile(
        0.25
    )

    Q3 = df[col].quantile(
        0.75
    )

    IQR = Q3 - Q1

    lower = (
        Q1 - 1.5 * IQR
    )

    upper = (
        Q3 + 1.5 * IQR
    )

    df[col] = df[col].clip(
        lower=lower,
        upper=upper
    )

    print(
        col,
        "-> IQR clipping applied."
    )





========== OUTLIER TREATMENT ==========
price -> IQR clipping applied.
freight_value -> IQR clipping applied.


In [29]:
# ============================================================
# 28. SKEWNESS SOLUTION
# ============================================================

print("\n========== SKEWNESS SOLUTION ==========")

# Re-check after outlier treatment

skew_after_outlier = df[
    continuous_cols
].skew()

print(
    "Skewness after outlier treatment:"
)

print(skew_after_outlier)


# Apply Yeo-Johnson only to highly skewed variables

skewed_cols = [
    col
    for col in continuous_cols
    if abs(
        skew_after_outlier[col]
    ) > 1
]

print(
    "\nVariables selected for transformation:"
)

print(skewed_cols)


if len(skewed_cols) > 0:

    transformer = PowerTransformer(
        method="yeo-johnson",
        standardize=False
    )

    df[skewed_cols] = (
        transformer.fit_transform(
            df[skewed_cols]
        )
    )

    print(
        "Yeo-Johnson transformation applied."
    )

else:

    print(
        "No highly skewed continuous variable requires transformation."
    )





========== SKEWNESS SOLUTION ==========
Skewness after outlier treatment:
price            1.084384
freight_value    0.699501
dtype: float64

Variables selected for transformation:
['price']
Yeo-Johnson transformation applied.


In [30]:
# ============================================================
# 29. SKEWNESS AFTER SOLUTION
# ============================================================

print("\n========== SKEWNESS AFTER SOLUTION ==========")

skew_after = df[
    continuous_cols
].skew()

print(skew_after)





========== SKEWNESS AFTER SOLUTION ==========
price           -0.024595
freight_value    0.699501
dtype: float64


In [31]:
# ============================================================
# 30. KURTOSIS AFTER SOLUTION
# ============================================================

print("\n========== KURTOSIS AFTER SOLUTION ==========")

kurt_after = df[
    continuous_cols
].kurt()

print(kurt_after)





========== KURTOSIS AFTER SOLUTION ==========
price           -0.693601
freight_value   -0.077896
dtype: float64


In [32]:
# ============================================================
# 31. NORMALITY TEST AFTER SOLUTION
# ============================================================

print("\n========== NORMALITY TEST AFTER SOLUTION ==========")

normality_after = []

normal_sample_after = df[
    continuous_cols
].sample(
    min(5000, len(df)),
    random_state=42
)

for col in continuous_cols:

    values = (
        normal_sample_after[col]
        .dropna()
    )

    statistic, p = normaltest(
        values
    )

    normality_after.append([
        col,
        statistic,
        p
    ])

    if p < 0.05:

        result = (
            "Not normally distributed"
        )

    else:

        result = (
            "Approximately normal"
        )

    print(
        f"{col}: p={p:.6f} -> {result}"
    )


normality_after_df = pd.DataFrame(
    normality_after,
    columns=[
        "Column",
        "Statistic",
        "P-value"
    ]
)





========== NORMALITY TEST AFTER SOLUTION ==========
price: p=0.000000 -> Not normally distributed
freight_value: p=0.000000 -> Not normally distributed


In [33]:
# ============================================================
# 32. UNIVARIATE ANALYSIS
# ============================================================

print("\n========== UNIVARIATE ANALYSIS ==========")


# PRICE HISTOGRAM

plt.figure(
    figsize=(8, 5)
)

sns.histplot(
    df["price"],
    kde=True
)

plt.title(
    "Price Distribution"
)

plt.xlabel(
    "Price"
)

plt.ylabel(
    "Frequency"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/price_histogram.png",
    dpi=300
)

plt.close()


# FREIGHT HISTOGRAM

plt.figure(
    figsize=(8, 5)
)

sns.histplot(
    df["freight_value"],
    kde=True
)

plt.title(
    "Freight Value Distribution"
)

plt.xlabel(
    "Freight Value"
)

plt.ylabel(
    "Frequency"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/freight_histogram.png",
    dpi=300
)

plt.close()


# PRICE BOXPLOT

plt.figure(
    figsize=(8, 5)
)

sns.boxplot(
    x=df["price"]
)

plt.title(
    "Price Boxplot After Outlier Treatment"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/price_boxplot.png",
    dpi=300
)

plt.close()


# FREIGHT BOXPLOT

plt.figure(
    figsize=(8, 5)
)

sns.boxplot(
    x=df["freight_value"]
)

plt.title(
    "Freight Value Boxplot After Outlier Treatment"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/freight_boxplot.png",
    dpi=300
)

plt.close()





========== UNIVARIATE ANALYSIS ==========


In [34]:
# ============================================================
# 33. BIVARIATE ANALYSIS
# ============================================================

print("\n========== BIVARIATE ANALYSIS ==========")

sample = df.sample(
    min(10000, len(df)),
    random_state=42
)

plt.figure(
    figsize=(8, 5)
)

sns.scatterplot(
    data=sample,
    x="price",
    y="freight_value",
    alpha=0.5
)

plt.title(
    "Price vs Freight Value"
)

plt.xlabel(
    "Price"
)

plt.ylabel(
    "Freight Value"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/price_vs_freight.png",
    dpi=300
)

plt.close()





========== BIVARIATE ANALYSIS ==========


In [35]:
# ============================================================
# 34. MULTIVARIATE ANALYSIS
# ============================================================

print("\n========== MULTIVARIATE ANALYSIS ==========")

pair_data = df[
    [
        "price",
        "freight_value",
        "order_item_id",
        "order_status"
    ]
].dropna()

pair_data = pair_data.sample(
    min(3000, len(pair_data)),
    random_state=42
)

pair_plot = sns.pairplot(
    pair_data,
    hue="order_status"
)

pair_plot.savefig(
    "EDA_Plots/pairplot.png",
    dpi=300
)

plt.close("all")





========== MULTIVARIATE ANALYSIS ==========


In [36]:
# ============================================================
# 35. CORRELATION
# ============================================================

print("\n========== CORRELATION ==========")

corr = df[
    numeric_cols
].corr()

print(corr)

plt.figure(
    figsize=(10, 8)
)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title(
    "Correlation Heatmap"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/correlation_heatmap.png",
    dpi=300
)

plt.close()





========== CORRELATION ==========
                          customer_zip_code_prefix  order_item_id     price  \
customer_zip_code_prefix                  1.000000      -0.000713  0.058497   
order_item_id                            -0.000713       1.000000 -0.106643   
price                                     0.058497      -0.106643  1.000000   
freight_value                             0.344511      -0.049000  0.429844   

                          freight_value  
customer_zip_code_prefix       0.344511  
order_item_id                 -0.049000  
price                          0.429844  
freight_value                  1.000000  


In [37]:
# ============================================================
# 36. ORDER STATUS VISUALIZATION
# ============================================================

print(
    "\n========== ORDER STATUS VISUALIZATION =========="
)

plt.figure(
    figsize=(8, 5)
)

sns.countplot(
    data=df,
    x="order_status",
    order=df[
        "order_status"
    ].value_counts().index
)

plt.title(
    "Order Status Distribution"
)

plt.xlabel(
    "Order Status"
)

plt.ylabel(
    "Number of Records"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/order_status_countplot.png",
    dpi=300
)

plt.close()





========== ORDER STATUS VISUALIZATION ==========


In [38]:
# ============================================================
# 37. TOP CUSTOMER STATES
# ============================================================

print(
    "\n========== TOP CUSTOMER STATES =========="
)

top_states = (
    df[
        "customer_state"
    ]
    .value_counts()
    .head(15)
)

plt.figure(
    figsize=(10, 6)
)

sns.barplot(
    x=top_states.values,
    y=top_states.index
)

plt.title(
    "Top 15 Customer States"
)

plt.xlabel(
    "Number of Records"
)

plt.ylabel(
    "State"
)

plt.tight_layout()

plt.savefig(
    "EDA_Plots/top_states.png",
    dpi=300
)

plt.close()





========== TOP CUSTOMER STATES ==========


In [39]:
# ============================================================
# 38. FINAL CHECK
# ============================================================

print(
    "\n========== FINAL EDA CHECK =========="
)

print(
    "Remaining missing values:",
    df.isnull().sum().sum()
)

print(
    "Remaining duplicate rows:",
    df.duplicated().sum()
)

print(
    "Final Shape:",
    df.shape
)





========== FINAL EDA CHECK ==========
Remaining missing values: 0
Remaining duplicate rows: 0
Final Shape: (113425, 18)


In [40]:
# ============================================================
# 39. FINAL FINDINGS
# ============================================================

print("\n" + "=" * 70)
print("FINAL EDA FINDINGS")
print("=" * 70)

print("""
1. Dataset structure and dimensions were analyzed.

2. Missing values were identified and treated using
   median for numerical variables and mode for
   categorical variables.

3. Blank values were checked.

4. Complete duplicate rows were checked and removed
   only if present.

5. Duplicate IDs were analyzed separately.

6. Unique values and categorical frequencies were checked.

7. Descriptive statistics including mean, median,
   mode, variance and standard deviation were calculated.

8. Grouping and comparison of means were performed.

9. Levene test was used to check variance homogeneity.

10. Chi-Square test was used to check independence/
    association between categorical variables.

11. Cramer's V was calculated to understand the strength
    of the categorical association.

12. D'Agostino-Pearson normality test was applied to
    check whether numerical variables follow a normal
    distribution.

13. Skewness was checked before and after treatment.

14. IQR method was used to detect continuous-variable
    outliers.

15. Outliers in price and freight_value were treated
    using IQR clipping.

16. Highly skewed continuous variables were treated
    using Yeo-Johnson transformation.

17. Kurtosis was checked before and after treatment.

18. Univariate analysis was performed.

19. Bivariate analysis was performed.

20. Multivariate analysis was performed.

21. Correlation analysis was performed.

22. PNG visualizations were generated and saved.

23. NO categorical encoding was performed.

24. NO feature scaling was performed.

25. NO train/test split was performed.

26. NO machine learning model was trained.

27. This script is EDA and data-cleaning/preparation only.
""")





FINAL EDA FINDINGS

1. Dataset structure and dimensions were analyzed.

2. Missing values were identified and treated using
   median for numerical variables and mode for
   categorical variables.

3. Blank values were checked.

4. Complete duplicate rows were checked and removed
   only if present.

5. Duplicate IDs were analyzed separately.

6. Unique values and categorical frequencies were checked.

7. Descriptive statistics including mean, median,
   mode, variance and standard deviation were calculated.

8. Grouping and comparison of means were performed.

9. Levene test was used to check variance homogeneity.

10. Chi-Square test was used to check independence/
    association between categorical variables.

11. Cramer's V was calculated to understand the strength
    of the categorical association.

12. D'Agostino-Pearson normality test was applied to
    check whether numerical variables follow a normal
    distribution.

13. Skewness was checked before and after treatment.

1

In [41]:
# ============================================================
# 40. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)

print(
    "COMPLETE EDA FINISHED SUCCESSFULLY!"
)

print("=" * 70)

print(
    "\nAll PNG plots are saved inside:"
)

print(
    os.path.abspath(
        "EDA_Plots"
    )
)

print(
    "\nEDA completed."
)


COMPLETE EDA FINISHED SUCCESSFULLY!

All PNG plots are saved inside:
c:\Users\Jannat\Documents\Project-First.py\First-Project\EDA_Plots

EDA completed.
